# ThreatLens AI — Notebook 03: Multi-Class Attack Classification

**Stage in the pipeline:** `AI Engines → Attack Classification`
**Input:** `data/processed/cicids2017_cleaned.parquet` (Notebook 01)

### What this notebook does
1. Loads the cleaned dataset
2. Trains a **Random Forest** classifier (baseline)
3. Trains an **XGBoost** classifier (comparison model)
4. Evaluates both — full per-class precision/recall/F1, not just overall accuracy
5. Compares them head-to-head and picks a production model with a stated reason
6. Looks at feature importance — which traffic characteristics actually drive each prediction
7. Saves the winning model

### Why this notebook exists separately from Notebook 02
Notebook 02's Isolation Forest can only say "this looks unusual." It has no idea *what* the unusual thing is. This notebook fixes that gap: given a row, it names the specific attack type — Brute Force, DDoS, Port Scanning, Web Attack, Botnet, or Infiltration — which is what actually lets a SOC analyst decide *how* to respond.


In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from src.models.anomaly import select_feature_columns
from src.models.classifier import (
    train_random_forest, train_xgboost, evaluate_classifier,
    get_feature_importance, save_model,
)

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (10, 5)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models" / "classifier"


## 1. Load data and prepare features/target

We reuse `select_feature_columns()` from `src/models/anomaly.py` — the **exact same feature set** used for the anomaly detector in Notebook 02. This is intentional: if the classifier outperforms the anomaly detector, we want that difference to come from the algorithm, not from the classifier secretly getting better inputs.

The target this time is `attack_category` itself, not just a binary "attack or not" flag — this is what makes this a genuinely *supervised, multi-class* problem.


In [2]:
df = pd.read_parquet(PROCESSED_DIR / "cicids2017_cleaned.parquet")
feature_cols = select_feature_columns(df)

X = df[feature_cols]
y = df["attack_category"]

print(f"Features: {len(feature_cols)} | Rows: {len(df):,}")
print(y.value_counts())

Features: 78 | Rows: 2,572,640
attack_category
Normal           2146899
DDoS              321759
Port Scanning      90694
Brute Force         9150
Web Attack          2143
Botnet              1948
Infiltration          36
Other                 11
Name: count, dtype: int64


## 2. Train/test split

Same `random_state=42` and same `stratify=` approach as Notebook 02, so results across notebooks stay comparable. Stratifying matters even more here than for the binary case — without it, a random split could easily leave a rare class like Infiltration with zero rows in the test set, making it impossible to evaluate.


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print("\nClass balance preserved in test set:")
print(y_test.value_counts(normalize=True).round(3))

Train: 1,929,480 | Test: 643,160

Class balance preserved in test set:
attack_category
Normal           0.835
DDoS             0.125
Port Scanning    0.035
Brute Force      0.004
Web Attack       0.001
Botnet           0.001
Infiltration     0.000
Other            0.000
Name: proportion, dtype: float64


## 3. Model 1 — Random Forest (baseline)

`class_weight="balanced"` tells the forest to pay proportionally more attention to rare classes during training, instead of optimizing purely for overall accuracy (which a lazy model could win just by always predicting "Normal" or "DDoS").


In [ ]:
rf_model = train_random_forest(X_train, y_train)
rf_pred = rf_model.predict(X_test)

label_order = sorted(y.unique())
rf_results = evaluate_classifier(y_test, rf_pred, label_order)
print(f"\nRandom Forest — Accuracy: {rf_results['accuracy']:.3f} | "
      f"Macro F1: {rf_results['macro_f1']:.3f} | Weighted F1: {rf_results['weighted_f1']:.3f}")

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(rf_results["confusion_matrix"], annot=True, fmt="d", cmap="mako",
            xticklabels=label_order, yticklabels=label_order)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.xticks(rotation=35, ha="right")
plt.title("Random Forest — Confusion Matrix")
plt.tight_layout()
plt.show()

## 4. Model 2 — XGBoost (comparison)

XGBoost requires integer-encoded labels rather than strings, so we fit a `LabelEncoder` here — kept local to this notebook (not inside `src/models/classifier.py`) since encoding is a training-time convenience, not a reusable modeling decision.


In [ ]:
encoder = LabelEncoder()
y_train_enc = encoder.fit_transform(y_train)
y_test_enc = encoder.transform(y_test)

xgb_model = train_xgboost(X_train, y_train_enc, num_classes=len(encoder.classes_))
xgb_pred_enc = xgb_model.predict(X_test)
xgb_pred = encoder.inverse_transform(xgb_pred_enc)

xgb_results = evaluate_classifier(y_test, xgb_pred, label_order)
print(f"\nXGBoost — Accuracy: {xgb_results['accuracy']:.3f} | "
      f"Macro F1: {xgb_results['macro_f1']:.3f} | Weighted F1: {xgb_results['weighted_f1']:.3f}")

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(xgb_results["confusion_matrix"], annot=True, fmt="d", cmap="rocket",
            xticklabels=label_order, yticklabels=label_order)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.xticks(rotation=35, ha="right")
plt.title("XGBoost — Confusion Matrix")
plt.tight_layout()
plt.show()

## 5. Head-to-head comparison

We compare on **macro F1**, not accuracy, as the deciding metric — macro F1 treats every class equally regardless of how many rows it has, so a model can't win just by nailing the huge "Normal"/"DDoS" classes while ignoring rare ones like Infiltration. This is exactly the kind of transparent, evidence-based selection the blueprint's "Model Lab" page is meant to represent.


In [ ]:
comparison = pd.DataFrame({
    "Random Forest": [rf_results["accuracy"], rf_results["macro_f1"], rf_results["weighted_f1"]],
    "XGBoost":       [xgb_results["accuracy"], xgb_results["macro_f1"], xgb_results["weighted_f1"]],
}, index=["Accuracy", "Macro F1", "Weighted F1"])

print(comparison.round(4))

winner = "XGBoost" if xgb_results["macro_f1"] > rf_results["macro_f1"] else "Random Forest"
print(f"\nSelected production model: {winner}  (higher macro F1 = better balanced performance across all attack types)")

## 6. Feature importance

Which traffic characteristics actually drive the winning model's decisions? This is a first, coarse layer of explainability — full per-prediction explanations (via SHAP) come in the next notebook, which explains not just "what matters overall" but "what mattered for *this specific* alert."


In [ ]:
winning_model = xgb_model if winner == "XGBoost" else rf_model
importance = get_feature_importance(winning_model, feature_cols)

print(importance.head(15))

plt.figure(figsize=(9, 6))
top15 = importance.head(15)
sns.barplot(x=top15.values, y=top15.index, hue=top15.index, palette="mako", legend=False)
plt.xlabel("Importance")
plt.title(f"Top 15 features — {winner}")
plt.tight_layout()
plt.show()

## 7. Save the winning model

Along with the model itself, we save the `LabelEncoder` if XGBoost won — the FastAPI inference service will need it later to turn the model's raw integer predictions back into readable attack names.


In [ ]:
save_model(winning_model, MODELS_DIR / f"attack_classifier_{winner.lower().replace(' ', '_')}_v1.joblib")
if winner == "XGBoost":
    save_model(encoder, MODELS_DIR / "label_encoder_v1.joblib")

## 8. Summary & next steps

| Item | Result |
|---|---|
| Models compared | Random Forest vs XGBoost |
| Selected model | see Section 5 output above |
| Macro F1 (selected) | see Section 5 output above |
| Weakest classes | check per-class recall in Section 3/4 classification reports above |
| Top driving features | see Section 6 |
| Saved model | `models/classifier/attack_classifier_*_v1.joblib` |

**Honest note:** whichever model wins here, check the per-class report above for classes with low recall — those are the attack types most likely to slip through in production, and worth flagging to a human reviewer rather than assuming the model has it covered.

**Next notebook (`04_shap_explainability.ipynb`)** takes the winning classifier from this notebook and explains individual predictions with SHAP — turning "the model says Brute Force, 91% risk" into "...specifically because of a failed-login spike, a new IP, and an off-hours login," matching the SHAP page already built into the dashboard frontend.
